# AI-Based Delivery Delay Risk Dataset Generation
This notebook creates a synthetic dataset for modeling multi-class delivery delay risk in e-commerce logistics.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import numpy as np
import pandas as pd


In [ ]:
# Set random seed for reproducibility
# Ensures the same dataset can be regenerated if needed
np.random.seed(96)


In [ ]:

# ------------------------------------------------------------
# STEP 1: DEFINE DATASET SIZE
# ------------------------------------------------------------
# Total number of delivery records (orders)
n = 20000


In [ ]:

# ------------------------------------------------------------
# STEP 2: GENERATE RAW FEATURES
# ------------------------------------------------------------
# These features are inspired by real-world supply chain and
# logistics operations.

data = pd.DataFrame({
    # Number of orders handled by the warehouse on that day
    # Higher volume increases processing and dispatch delays
    "order_volume": np.random.randint(10, 500, n),

    # Time taken by the warehouse to process the order (in hours)
    # Includes picking, packing, and dispatch preparation
    "warehouse_time_hrs": np.random.uniform(1, 48, n),

    # Distance between warehouse and customer delivery location (in km)
    "shipment_distance_km": np.random.uniform(5, 3000, n),

    # Traffic congestion level on delivery route
    # 1 = Low traffic, 5 = Severe congestion
    "traffic_level": np.random.randint(1, 6, n),

    # Weather impact severity
    # 0 = Clear, 1 = Mild, 2 = Moderate, 3 = Severe
    "weather_severity": np.random.randint(0, 4, n),

    # Utilization level of delivery personnel (in percentage)
    # Higher load indicates stretched delivery capacity
    "courier_load_pct": np.random.uniform(30, 100, n),

    # Historical delay rate for the delivery route
    # Represents past reliability of the route
    "past_delay_rate": np.random.uniform(0.0, 0.6, n)
})
print(data.head())

   order_volume  warehouse_time_hrs  shipment_distance_km  traffic_level  \
0            94            4.314205           2307.511608              2   
1           484           26.992300           1721.851885              5   
2           432           37.363071           1919.134927              4   
3           374            2.536696            373.240260              3   
4           126            8.527455           1593.551859              5   

   weather_severity  courier_load_pct  past_delay_rate  
0                 0         45.789611         0.332918  
1                 3         36.217764         0.326364  
2                 0         86.955847         0.070964  
3                 1         85.850780         0.328125  
4                 3         39.788405         0.430941  


In [ ]:

# ------------------------------------------------------------
# STEP 3: NORMALIZE FEATURES
# ------------------------------------------------------------
# Normalization ensures that features with larger numeric
# ranges do not dominate the risk score unfairly.

data["order_volume_norm"] = data["order_volume"] / data["order_volume"].max()
data["warehouse_time_norm"] = data["warehouse_time_hrs"] / data["warehouse_time_hrs"].max()
data["distance_norm"] = data["shipment_distance_km"] / data["shipment_distance_km"].max()
data["traffic_norm"] = data["traffic_level"] / 5
data["weather_norm"] = data["weather_severity"] / 3
data["courier_load_norm"] = data["courier_load_pct"] / 100
data["past_delay_norm"] = data["past_delay_rate"] / 0.6


In [ ]:

# ------------------------------------------------------------
# STEP 4: COMPUTE DELIVERY RISK SCORE
# ------------------------------------------------------------
# A weighted risk score is calculated using domain intuition:
# - Traffic and weather have the strongest impact
# - Historical delays strongly influence future outcomes
# - Distance, warehouse time, and courier load have moderate impact
# - Order volume has relatively lower influence

data["risk_score"] = (
    0.10 * data["order_volume_norm"] +
    0.15 * data["warehouse_time_norm"] +
    0.15 * data["distance_norm"] +
    0.20 * data["traffic_norm"] +
    0.15 * data["weather_norm"] +
    0.10 * data["courier_load_norm"] +
    0.15 * data["past_delay_norm"]
)


In [ ]:

# ------------------------------------------------------------
# STEP 5: ADD CONTROLLED STOCHASTIC NOISE
# ------------------------------------------------------------
# Real-world logistics are affected by unpredictable factors
# such as sudden traffic jams, minor accidents, or system delays.
# Small noise is added to simulate this uncertainty.

noise = np.random.uniform(-0.05, 0.05, n)
data["risk_score"] = data["risk_score"] + noise

# Ensure risk score remains within valid bounds
data["risk_score"] = data["risk_score"].clip(0, 1)


In [ ]:

# ------------------------------------------------------------
# STEP 6: ASSIGN MULTI-CLASS DELIVERY STATUS
# ------------------------------------------------------------
# Instead of hard thresholds, probabilistic boundaries are used
# near class edges to avoid deterministic labeling.

def assign_delivery_status(risk):
    if risk < 0.45:
        return "On-Time"
    elif 0.45 <= risk <= 0.55:
        return np.random.choice(["On-Time", "At Risk"], p=[0.6, 0.4])
    elif 0.55 < risk < 0.70:
        return "At Risk"
    elif 0.70 <= risk <= 0.80:
        return np.random.choice(["At Risk", "Delayed"], p=[0.7, 0.3])
    else:
        return "Delayed"

# Apply classification logic
data["delivery_status"] = data["risk_score"].apply(assign_delivery_status)


In [ ]:

# ------------------------------------------------------------
# STEP 7: CLEANUP INTERMEDIATE COLUMNS
# ------------------------------------------------------------
# Remove normalized helper columns to keep the dataset clean
# and realistic for downstream machine learning tasks.

data.drop(
    columns=[
        "order_volume_norm",
        "warehouse_time_norm",
        "distance_norm",
        "traffic_norm",
        "weather_norm",
        "courier_load_norm",
        "past_delay_norm"
    ],
    inplace=True
)


In [ ]:

# ------------------------------------------------------------
# STEP 8: QUICK DATASET VALIDATION
# ------------------------------------------------------------

# View sample records
print(data.head())

# Check class distribution (should be naturally imbalanced)
data["delivery_status"].value_counts()

# Confirm dataset dimensions
data.shape


   order_volume  warehouse_time_hrs  shipment_distance_km  traffic_level  \
0            94            4.314205           2307.511608              2   
1           484           26.992300           1721.851885              5   
2           432           37.363071           1919.134927              4   
3           374            2.536696            373.240260              3   
4           126            8.527455           1593.551859              5   

   weather_severity  courier_load_pct  past_delay_rate  risk_score  \
0                 0         45.789611         0.332918    0.332560   
1                 3         36.217764         0.326364    0.731490   
2                 0         86.955847         0.070964    0.551166   
3                 1         85.850780         0.328125    0.413695   
4                 3         39.788405         0.430941    0.622663   

  delivery_status  
0         On-Time  
1         At Risk  
2         At Risk  
3         On-Time  
4         At Risk  


(20000, 9)

In [ ]:

# ------------------------------------------------------------
# OPTIONAL: SAVE DATASET TO CSV
# ------------------------------------------------------------
# Uncomment if you want to persist the dataset
data.to_csv("delivery_delay_dataset.csv", index=False)

In [ ]:
from google.colab import files
files.download("delivery_delay_dataset.csv")

## Introduce New Features

### Subtask:
Add features such as 'day_of_week', 'is_peak_hour', 'warehouse_region', and 'delivery_type' to the dataset.


In [ ]:
n = len(data)

# 1. Generate 'day_of_week' (0=Monday to 6=Sunday)
data["day_of_week"] = np.random.randint(0, 7, n)

# 2. Generate 'is_peak_hour' (0 or 1)
data["is_peak_hour"] = np.random.randint(0, 2, n)

# 3. Generate 'warehouse_region'
regions = ['North', 'South', 'East', 'West', 'Central']
data["warehouse_region"] = np.random.choice(regions, n)

# 4. Generate 'delivery_type'
delivery_types = ['Standard', 'Express', 'Same-Day']
data["delivery_type"] = np.random.choice(delivery_types, n)

print("New features added to the dataset:")
print(data[['day_of_week', 'is_peak_hour', 'warehouse_region', 'delivery_type']].head())

New features added to the dataset:
   day_of_week  is_peak_hour warehouse_region delivery_type
0            6             1             West      Same-Day
1            3             1          Central       Express
2            2             0            North       Express
3            6             0            North      Same-Day
4            0             1             West      Standard


# DATA CREATION FROM SCRATCH (Not Required)

In [ ]:
# import numpy as np
# import pandas as pd

# # Set random seed for reproducibility (from cell nwsmXRfrY1Gi)
# np.random.seed(42)

# # Define dataset size (from cell K_o9sAn4Y2dK)
# n = 20000

# # STEP 2: GENERATE RAW FEATURES (from cell DL7hVTMjY4Qf)
# data = pd.DataFrame({
#     "order_volume": np.random.randint(10, 500, n),
#     "warehouse_time_hrs": np.random.uniform(1, 48, n),
#     "shipment_distance_km": np.random.uniform(5, 3000, n),
#     "traffic_level": np.random.randint(1, 6, n),
#     "weather_severity": np.random.randint(0, 4, n),
#     "courier_load_pct": np.random.uniform(30, 100, n),
#     "past_delay_rate": np.random.uniform(0.0, 0.6, n)
# })

# # STEP 3: NORMALIZE FEATURES (from cell 2loBhyQHY5zP)
# data["order_volume_norm"] = data["order_volume"] / data["order_volume"].max()
# data["warehouse_time_norm"] = data["warehouse_time_hrs"] / data["warehouse_time_hrs"].max()
# data["distance_norm"] = data["shipment_distance_km"] / data["shipment_distance_km"].max()
# data["traffic_norm"] = data["traffic_level"] / 5
# data["weather_norm"] = data["weather_severity"] / 3
# data["courier_load_norm"] = data["courier_load_pct"] / 100
# data["past_delay_norm"] = data["past_delay_rate"] / 0.6

# # STEP 4: COMPUTE DELIVERY RISK SCORE (from cell uhPHu4swY6xN)
# data["risk_score"] = (
#     0.10 * data["order_volume_norm"] +
#     0.15 * data["warehouse_time_norm"] +
#     0.15 * data["distance_norm"] +
#     0.20 * data["traffic_norm"] +
#     0.15 * data["weather_norm"] +
#     0.10 * data["courier_load_norm"] +
#     0.15 * data["past_delay_norm"]
# )

# # STEP 5: ADD CONTROLLED STOCHASTIC NOISE (from cell ru4bXVwaY77X)
# noise = np.random.uniform(-0.05, 0.05, n)
# data["risk_score"] = data["risk_score"] + noise
# data["risk_score"] = data["risk_score"].clip(0, 1)

# # STEP 6: ASSIGN MULTI-CLASS DELIVERY STATUS (from cell KfVUVNt_Y9oc)
# def assign_delivery_status(risk):
#     if risk < 0.45:
#         return "On-Time"
#     elif 0.45 <= risk <= 0.55:
#         return np.random.choice(["On-Time", "At Risk"], p=[0.6, 0.4])
#     elif 0.55 < risk < 0.70:
#         return "At Risk"
#     elif 0.70 <= risk <= 0.80:
#         return np.random.choice(["At Risk", "Delayed"], p=[0.7, 0.3])
#     else:
#         return "Delayed"
# data["delivery_status"] = data["risk_score"].apply(assign_delivery_status)

# # STEP 7: CLEANUP INTERMEDIATE COLUMNS (from cell InDmfwAHY-qu)
# data.drop(
#     columns=[
#         "order_volume_norm",
#         "warehouse_time_norm",
#         "distance_norm",
#         "traffic_norm",
#         "weather_norm",
#         "courier_load_norm",
#         "past_delay_norm"
#     ],
#     inplace=True
# )

# # Now, add the new features as originally intended in the subtask
# # n is already defined above and matches the length of data.

# # 1. Generate 'day_of_week' (0=Monday to 6=Sunday)
# data["day_of_week"] = np.random.randint(0, 7, n)

# # 2. Generate 'is_peak_hour' (0 or 1)
# data["is_peak_hour"] = np.random.randint(0, 2, n)

# # 3. Generate 'warehouse_region'
# regions = ['North', 'South', 'East', 'West', 'Central']
# data["warehouse_region"] = np.random.choice(regions, n)

# # 4. Generate 'delivery_type'
# delivery_types = ['Standard', 'Express', 'Same-Day']
# data["delivery_type"] = np.random.choice(delivery_types, n)

# print("New features added to the dataset:")
# print(data[['day_of_week', 'is_peak_hour', 'warehouse_region', 'delivery_type']].head())

## Enhance Risk Score Calculation

### Subtask:
Modify the 'risk_score' calculation to incorporate the newly introduced features: 'day_of_week', 'is_peak_hour', 'warehouse_region', and 'delivery_type'.


In [ ]:
n = len(data)

# 1. Re-normalize original numerical features
data["order_volume_norm"] = data["order_volume"] / data["order_volume"].max()
data["warehouse_time_norm"] = data["warehouse_time_hrs"] / data["warehouse_time_hrs"].max()
data["distance_norm"] = data["shipment_distance_km"] / data["shipment_distance_km"].max()
data["traffic_norm"] = data["traffic_level"] / 5
data["weather_norm"] = data["weather_severity"] / 3
data["courier_load_norm"] = data["courier_load_pct"] / 100
data["past_delay_norm"] = data["past_delay_rate"] / 0.6

# 2. Normalize new numerical features
data["day_of_week_norm"] = data["day_of_week"] / 6 # Max value is 6 (Sunday)
data["is_peak_hour_norm"] = data["is_peak_hour"] # Already 0 or 1

# 3. Create risk factors for new categorical features
warehouse_region_mapping = {'North': 0.9, 'South': 1.0, 'East': 1.1, 'West': 1.0, 'Central': 1.2}
delivery_type_mapping = {'Standard': 0.9, 'Express': 1.1, 'Same-Day': 1.3}

data["warehouse_region_risk_factor"] = data["warehouse_region"].map(warehouse_region_mapping)
data["delivery_type_risk_factor"] = data["delivery_type"].map(delivery_type_mapping)

# 4. Recalculate the 'risk_score' with new weights
data["risk_score"] = (
    0.08 * data["order_volume_norm"] +
    0.12 * data["warehouse_time_norm"] +
    0.12 * data["distance_norm"] +
    0.16 * data["traffic_norm"] +
    0.12 * data["weather_norm"] +
    0.08 * data["courier_load_norm"] +
    0.12 * data["past_delay_norm"] +
    0.05 * data["day_of_week_norm"] +
    0.05 * data["is_peak_hour_norm"] +
    0.05 * data["warehouse_region_risk_factor"] +
    0.05 * data["delivery_type_risk_factor"]
)

# 5. Add controlled stochastic noise
noise = np.random.uniform(-0.05, 0.05, n)
data["risk_score"] = data["risk_score"] + noise
data["risk_score"] = data["risk_score"].clip(0, 1) # Ensure risk score remains within valid bounds

# 6. Re-assign 'delivery_status'
# The assign_delivery_status function from cell KfVUVNt_Y9oc needs to be re-defined if not in scope.
# Assuming it's in scope or we re-define it for clarity.
def assign_delivery_status(risk):
    if risk < 0.45:
        return "On-Time"
    elif 0.45 <= risk <= 0.55:
        return np.random.choice(["On-Time", "At Risk"], p=[0.6, 0.4])
    elif 0.55 < risk < 0.70:
        return "At Risk"
    elif 0.70 <= risk <= 0.80:
        return np.random.choice(["At Risk", "Delayed"], p=[0.7, 0.3])
    else:
        return "Delayed"

data["delivery_status"] = data["risk_score"].apply(assign_delivery_status)

# 7. Clean up intermediate columns
data.drop(
    columns=[
        "order_volume_norm",
        "warehouse_time_norm",
        "distance_norm",
        "traffic_norm",
        "weather_norm",
        "courier_load_norm",
        "past_delay_norm",
        "day_of_week_norm",
        "is_peak_hour_norm",
        "warehouse_region_risk_factor",
        "delivery_type_risk_factor"
    ],
    inplace=True
)

print("Risk score and delivery status recalculated and updated.")
print(data[['risk_score', 'delivery_status', 'day_of_week', 'is_peak_hour', 'warehouse_region', 'delivery_type']].head())

Risk score and delivery status recalculated and updated.
   risk_score delivery_status  day_of_week  is_peak_hour warehouse_region  \
0    0.486537         On-Time            6             1             West   
1    0.745403         At Risk            3             1          Central   
2    0.590096         At Risk            2             0            North   
3    0.543604         At Risk            6             0            North   
4    0.600323         At Risk            0             1             West   

  delivery_type  
0      Same-Day  
1       Express  
2       Express  
3      Same-Day  
4      Standard  


## Adjust Delivery Status Assignment Logic

### Subtask:
Refine the 'assign_delivery_status' function by adjusting thresholds and probabilistic assignments to create clearer separation between classes.


**Reasoning**:
The subtask requires refining the 'assign_delivery_status' function with new thresholds and probabilities. I will define the new function, apply it to the 'data' DataFrame's 'risk_score' column to update 'delivery_status', and then display the value counts of the updated 'delivery_status' to show the class distribution.



In [ ]:
def assign_delivery_status_refined(risk):
    if risk < 0.42:
        return "On-Time"
    elif 0.42 <= risk <= 0.50:
        return np.random.choice(["On-Time", "At Risk"], p=[0.7, 0.3])
    elif 0.50 < risk < 0.72:
        return "At Risk"
    elif 0.72 <= risk <= 0.82:
        return np.random.choice(["At Risk", "Delayed"], p=[0.6, 0.4])
    else:
        return "Delayed"

# Apply the refined classification logic
data["delivery_status"] = data["risk_score"].apply(assign_delivery_status_refined)

# Display the value counts of the updated delivery status
print("Updated delivery_status class distribution:")
print(data["delivery_status"].value_counts())

Updated delivery_status class distribution:
delivery_status
At Risk    15931
On-Time     3195
Delayed      874
Name: count, dtype: int64


## Analyze Enhanced Dataset Characteristics

### Subtask:
Analyze the distributions of the new features and their correlation with 'delivery_status'.


**Reasoning**:
The subtask requires analyzing the distributions of new features and their correlation with 'delivery_status'. I will display value counts for numerical and categorical features, then calculate means for numerical features grouped by 'delivery_status', and finally create cross-tabulations for categorical features against 'delivery_status'.



In [ ]:
print("\n--- Distribution of 'day_of_week' ---")
print(data['day_of_week'].value_counts().sort_index())

print("\n--- Distribution of 'is_peak_hour' ---")
print(data['is_peak_hour'].value_counts().sort_index())

print("\n--- Distribution of 'warehouse_region' ---")
print(data['warehouse_region'].value_counts())

print("\n--- Distribution of 'delivery_type' ---")
print(data['delivery_type'].value_counts())

print("\n--- Mean of 'day_of_week' and 'is_peak_hour' grouped by 'delivery_status' ---")
print(data.groupby('delivery_status')[['day_of_week', 'is_peak_hour']].mean())

print("\n--- Cross-tabulation of 'warehouse_region' and 'delivery_status' ---")
print(pd.crosstab(data['warehouse_region'], data['delivery_status']))

print("\n--- Cross-tabulation of 'delivery_type' and 'delivery_status' ---")
print(pd.crosstab(data['delivery_type'], data['delivery_status']))


--- Distribution of 'day_of_week' ---
day_of_week
0    2887
1    2838
2    2828
3    2843
4    2882
5    2895
6    2827
Name: count, dtype: int64

--- Distribution of 'is_peak_hour' ---
is_peak_hour
0    10105
1     9895
Name: count, dtype: int64

--- Distribution of 'warehouse_region' ---
warehouse_region
West       4030
North      4022
East       4006
South      3977
Central    3965
Name: count, dtype: int64

--- Distribution of 'delivery_type' ---
delivery_type
Same-Day    6726
Express     6669
Standard    6605
Name: count, dtype: int64

--- Mean of 'day_of_week' and 'is_peak_hour' grouped by 'delivery_status' ---
                 day_of_week  is_peak_hour
delivery_status                           
At Risk             3.053920      0.516728
Delayed             3.673913      0.748284
On-Time             2.543036      0.315806

--- Cross-tabulation of 'warehouse_region' and 'delivery_status' ---
delivery_status   At Risk  Delayed  On-Time
warehouse_region                           
C

**Reasoning**:
The subtask requires visualizing the distributions of the new features and their correlation with 'delivery_status'. I will use `matplotlib.pyplot` and `seaborn` to create several plots: count plots for the distributions of each new feature and stacked bar plots to show the relationship between each new feature and the 'delivery_status'.



In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Set plot style
sns.set_style("whitegrid")

# --- Visualize Distributions of New Features ---
plt.figure(figsize=(18, 10))

plt.subplot(2, 2, 1)
sns.countplot(data=data, x='day_of_week', palette='viridis')
plt.title('Distribution of Day of Week')
plt.xlabel('Day of Week (0=Monday, 6=Sunday)')
plt.ylabel('Count')

plt.subplot(2, 2, 2)
sns.countplot(data=data, x='is_peak_hour', palette='plasma')
plt.title('Distribution of Peak Hour')
plt.xlabel('Is Peak Hour (0=No, 1=Yes)')
plt.ylabel('Count')

plt.subplot(2, 2, 3)
sns.countplot(data=data, x='warehouse_region', palette='magma')
plt.title('Distribution of Warehouse Region')
plt.xlabel('Warehouse Region')
plt.ylabel('Count')

plt.subplot(2, 2, 4)
sns.countplot(data=data, x='delivery_type', palette='cividis')
plt.title('Distribution of Delivery Type')
plt.xlabel('Delivery Type')
plt.ylabel('Count')

plt.tight_layout()
plt.show()

# --- Visualize Correlation of New Features with Delivery Status ---
plt.figure(figsize=(18, 12))

# day_of_week vs delivery_status
plt.subplot(2, 2, 1)
sns.countplot(data=data, x='day_of_week', hue='delivery_status', palette='viridis')
plt.title('Delivery Status by Day of Week')
plt.xlabel('Day of Week (0=Monday, 6=Sunday)')
plt.ylabel('Count')
plt.legend(title='Delivery Status', bbox_to_anchor=(1.05, 1), loc='upper left')

# is_peak_hour vs delivery_status
plt.subplot(2, 2, 2)
sns.countplot(data=data, x='is_peak_hour', hue='delivery_status', palette='plasma')
plt.title('Delivery Status by Peak Hour')
plt.xlabel('Is Peak Hour (0=No, 1=Yes)')
plt.ylabel('Count')
plt.legend(title='Delivery Status', bbox_to_anchor=(1.05, 1), loc='upper left')

# warehouse_region vs delivery_status
plt.subplot(2, 2, 3)
sns.countplot(data=data, x='warehouse_region', hue='delivery_status', palette='magma')
plt.title('Delivery Status by Warehouse Region')
plt.xlabel('Warehouse Region')
plt.ylabel('Count')
plt.legend(title='Delivery Status', bbox_to_anchor=(1.05, 1), loc='upper left')

# delivery_type vs delivery_status
plt.subplot(2, 2, 4)
sns.countplot(data=data, x='delivery_type', hue='delivery_status', palette='cividis')
plt.title('Delivery Status by Delivery Type')
plt.xlabel('Delivery Type')
plt.ylabel('Count')
plt.legend(title='Delivery Status', bbox_to_anchor=(1.05, 1), loc='upper left')

plt.tight_layout()
plt.show()


## Visualize Dataset for Insights

### Subtask:
Create histograms for continuous numerical features and visualize the distribution of 'risk_score' across different 'delivery_status' categories.


**Reasoning**:
The subtask requires visualizing the distributions of continuous numerical features and the 'risk_score' across different 'delivery_status' categories. I will create histograms for the specified numerical features and a violin plot for 'risk_score' by 'delivery_status' using `matplotlib` and `seaborn`.



In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Set plot style
sns.set_style("whitegrid")

# --- Create Histograms for Continuous Numerical Features ---

# Identify numerical features to plot
numerical_features = [
    'order_volume',
    'warehouse_time_hrs',
    'shipment_distance_km',
    'traffic_level',
    'weather_severity',
    'courier_load_pct',
    'past_delay_rate',
    'risk_score'
]

plt.figure(figsize=(16, 18))

for i, feature in enumerate(numerical_features):
    plt.subplot(4, 2, i + 1) # 4 rows, 2 columns
    sns.histplot(data=data, x=feature, kde=True, bins=30, palette='viridis')
    plt.title(f'Distribution of {feature.replace("_", " ").title()}')
    plt.xlabel(feature.replace("_", " ").title())
    plt.ylabel('Frequency')

plt.tight_layout()
plt.show()

# --- Visualize Distribution of 'risk_score' across 'delivery_status' categories ---
plt.figure(figsize=(10, 7))
sns.violinplot(data=data, x='delivery_status', y='risk_score', palette='coolwarm')
plt.title('Risk Score Distribution by Delivery Status')
plt.xlabel('Delivery Status')
plt.ylabel('Risk Score')
plt.show()

In [ ]:
data.to_csv("delivery_delay_dataset.csv", index=False)
print("Dataset saved to 'delivery_delay_dataset.csv'")

Dataset saved to 'delivery_delay_dataset.csv'


In [ ]:
from google.colab import files
files.download("delivery_delay_dataset.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Final Review (Daksh)


In [ ]:
print(data.info())

**Reasoning**:
To provide a comprehensive summary of all columns, I need to extract their descriptive statistics. For numerical columns, I'll calculate min, max, mean, and standard deviation. For categorical columns, I'll list unique values and their counts. This will give a complete overview of the data's characteristics.



In [ ]:
summary_report = {}

for col in data.columns:
    col_type = data[col].dtype
    summary_report[col] = {
        "dtype": str(col_type)
    }

    if pd.api.types.is_numeric_dtype(data[col]):
        summary_report[col]["min"] = data[col].min()
        summary_report[col]["max"] = data[col].max()
        summary_report[col]["mean"] = data[col].mean()
        summary_report[col]["std"] = data[col].std()
        if col_type == 'int64': # Special handling for integer types that are essentially categorical or ordinal
            if data[col].nunique() < 10: # If few unique values, treat as quasi-categorical
                 summary_report[col]["possible_values"] = data[col].unique().tolist()
            else: # Otherwise, it's a true numerical range
                 summary_report[col]["range"] = f"{data[col].min()} - {data[col].max()}"
    elif pd.api.types.is_object_dtype(data[col]):
        summary_report[col]["unique_values"] = data[col].unique().tolist()
        summary_report[col]["value_counts"] = data[col].value_counts().to_dict()


# Print the summary report in a readable format
print("### Comprehensive Data Dictionary and Summary")
for col, details in summary_report.items():
    print(f"\n--- Column: {col} ---")
    for key, value in details.items():
        print(f"  {key}: {value}")

## Summary:

### Q&A
The final dataset (`data` DataFrame) contains 20,000 entries and 13 columns. No missing values were found across any column. The data types include `int64`, `float64`, and `object`.

A comprehensive summary for each column is as follows:
*   **Numerical Columns (float64 and int64 with wide ranges)**:
    *   `order_volume` (int64): Ranges from 10 to 499.
    *   `warehouse_time_hrs` (float64): Ranges from approximately 1.0 to 48.0.
    *   `shipment_distance_km` (float64): Ranges from approximately 5.0 to 2999.9.
    *   `courier_load_pct` (float64): Ranges from approximately 30.0 to 99.9.
    *   `past_delay_rate` (float64): Ranges from approximately 0.00006 to 0.6.
    *   `risk_score` (float64): Ranges from approximately 0.21 to 0.94.
*   **Categorical/Ordinal Integer Columns (int64 with few unique values)**:
    *   `traffic_level`: Possible values are [2, 5, 4, 3, 1].
    *   `weather_severity`: Possible values are [0, 3, 1, 2].
    *   `day_of_week`: Possible values are [6, 3, 2, 0, 1, 4, 5].
    *   `is_peak_hour`: Possible values are [1, 0].
*   **Categorical String Columns (object)**:
    *   `delivery_status`: Unique values are 'On-Time', 'At Risk', 'Delayed'. The value counts are {'At Risk': 15931, 'On-Time': 3195, 'Delayed': 874}.
    *   `warehouse_region`: Unique values are 'West', 'Central', 'North', 'South', 'East', with relatively even distribution.
    *   `delivery_type`: Unique values are 'Same-Day', 'Express', 'Standard', with relatively even distribution.

### Data Analysis Key Findings
*   The dataset comprises 20,000 entries and 13 columns, with no missing values, ensuring data completeness for analysis.
*   The `delivery_status` column indicates that a significant majority of deliveries are 'At Risk' (15,931 out of 20,000), followed by 'On-Time' (3,195) and 'Delayed' (874).
*   Categorical variables like `traffic_level`, `weather_severity`, `day_of_week`, `is_peak_hour`, `warehouse_region`, and `delivery_type` have been identified with their respective distinct values, providing clear categories for segmentation and analysis.
*   Numerical columns, such as `order_volume` (range 10-499), `warehouse_time_hrs` (approx. 1.0-48.0), `shipment_distance_km` (approx. 5.0-2999.9), `courier_load_pct` (approx. 30.0-99.9), `past_delay_rate` (approx. 0.00006-0.6), and `risk_score` (approx. 0.21-0.94), provide a wide range of values that could influence delivery outcomes.

### Insights or Next Steps
*   Investigate the high proportion of 'At Risk' deliveries in the `delivery_status` column to understand the underlying factors and potential mitigation strategies.
*   Utilize the comprehensive data dictionary as a foundational reference for subsequent data cleaning, feature engineering, and predictive modeling tasks related to delivery performance.
